# Recurrent Refiner v2 - Treino no Kaggle

Base congelada (Qwen2.5-Coder-7B-Instruct, 4-bit) + refiner recorrente com halting adaptativo (ACT).

**Antes de rodar, no painel direito do notebook:**
1. **Accelerator** -> GPU T4 x2 (ou P100)
2. **Internet** -> On (precisa pra clonar o repo e baixar o modelo)
3. (Opcional, recomendado) **Add-ons -> Secrets** -> adicione um secret chamado `HF_TOKEN` com um token do HuggingFace (huggingface.co/settings/tokens, permissao de escrita) - permite subir/baixar checkpoints do Hub entre sessoes/plataformas.

Esta e uma run de **validacao (M1)**: poucos steps, so pra confirmar que o pipeline inteiro funciona (checkpoint sobe/desce do HF Hub certo, val loss cai). A run de treino de verdade (M2) vem depois, com mais steps.

In [ ]:
# Cell 1: Dependencias
# NAO reinstala o torch - o Kaggle ja vem com uma build de torch casada com o driver/CUDA da maquina;
# reinstalar por cima e uma causa classica de quebrar a compatibilidade com a GPU.
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub datasets

In [ ]:
# Cell 2: Clonar e instalar o pacote
!rm -rf recurrent-refiner
!git clone https://github.com/NullVoidDev/recurrent-refiner
%cd recurrent-refiner
!pip install -q -e .

In [ ]:
# Cell 3: Testar se o modulo importa + checar GPU
import torch
from recurrent_refiner import RefinerConfig, CodeRecurrentModel

print('CUDA disponivel:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)
print('OK')

In [ ]:
# Cell 4: Login no HuggingFace Hub (opcional, via Kaggle Secret HF_TOKEN)
# Sem isso, o treino ainda roda, so nao consegue subir/baixar checkpoint do Hub
# (cai de volta pra so salvar localmente em refiner_checkpoints/, que some quando a sessao acaba).
HUB_REPO_ID = 'keindev/recurrent-refiner-ckpts'

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Logado no HuggingFace Hub - checkpoints vao ser sincronizados com', HUB_REPO_ID)
except Exception as e:
    print(f'Sem HF_TOKEN configurado ({e}) - treino roda normalmente, so sem sync de checkpoint pelo Hub')

In [ ]:
# Cell 5: Run de validacao (M1) - poucos steps, so pra confirmar que tudo funciona
# Depois de validar, aumente --max_steps / --max_seq_len pra uma run de treino de verdade (M2).
!python -m recurrent_refiner.train \
  --base_model Qwen/Qwen2.5-Coder-7B-Instruct \
  --dataset google-research-datasets/mbpp \
  --max_steps 300 \
  --max_seq_len 512 \
  --save_every 100 \
  --eval_every 100 \
  --lr 5e-5 \
  --hub_repo_id {HUB_REPO_ID}

In [ ]:
# Cell 6: Testar o checkpoint treinado
# Nao precisa mais escolher n_loops - o halting adaptativo (ACT) decide sozinho quantos loops usar por token,
# entao nao existe mais o mismatch treino/inferencia que tinha na v1.
import glob, os, re, torch
from recurrent_refiner import RefinerConfig, CodeRecurrentModel
from transformers import AutoTokenizer

def checkpoint_step(path):
    m = re.search(r'step_(\d+)\.pt$', path)
    return int(m.group(1)) if m else -1

ckpts = glob.glob('refiner_checkpoints/*.pt')
if not ckpts:
    raise RuntimeError('Nenhum checkpoint encontrado. Rode a Cell 5 (treino) primeiro.')

best = [c for c in ckpts if os.path.basename(c) == 'best.pt']
step_ckpts = [c for c in ckpts if checkpoint_step(c) >= 0]
ckpt_path = best[0] if best else (max(step_ckpts, key=checkpoint_step) if step_ckpts else ckpts[-1])
print(f'Carregando: {ckpt_path}')

cfg = RefinerConfig(base_model_id='Qwen/Qwen2.5-Coder-7B-Instruct')
model = CodeRecurrentModel(cfg)

ckpt = torch.load(ckpt_path, map_location='cpu')
model.refiner.load_state_dict(ckpt['refiner'])
if 'lm_head' in ckpt:
    model.lm_head.load_state_dict(ckpt['lm_head'])

tok = AutoTokenizer.from_pretrained(cfg.base_model_id, trust_remote_code=True)

prompts = [
    'def quicksort(arr):',
    '# Write a function to check if a string is a palindrome',
    '# Implement a LRU cache',
]

for p in prompts:
    out = model.generate(tok, p, max_new=96)
    print(f'\nPrompt: {p}\n{out}\n{"-"*60}')